In [10]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [11]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [12]:
# h = load_queries("sql/02d_hotspots.sql")
# con.execute(h["create_zhw_cells"])
# con.execute(h["create_zhw_hotspots"])
# display(run(h, "cells_reconcile").T)

In [13]:
# display(run(h, "cell_size_distribution"))
# display(run(h, "cell_size_quantiles").T)
# sb = run(h, "slot_baseline")
# display(sb["pct_wait_ge10"].describe().round(3).to_frame("share of waits >= 10 min across the 168 slots"))
# display(sb.sort_values("pct_wait_ge10", ascending=False).head(8))

In [14]:
import pandas as pd
h = load_queries("sql/02d_hotspots.sql")
z = load_queries("sql/02d2_hotspots_zone_hour.sql")
try:
    con.execute("SELECT 1 FROM zhw_hotspots LIMIT 1")      # exists if you are still in the notebook 08 kernel
except Exception:
    con.execute(h["create_zhw_cells"]); con.execute(h["create_zhw_hotspots"])
for name in ["create_zh_cells", "create_zh_hotspots", "create_coverage_cells"]:
    con.execute(z[name])
display(run(z, "zh_cells_reconcile").T)

,0
n_cells,6180.0
n_zones,260.0
n_hours,24.0
empty_cells,60.0
cells_with_no_valid_wait,0.0
n_scoped,20404892.0
n_valid,20205118.0
n_excluded,199774.0


In [15]:
cov = run(z, "coverage_tradeoff")
display(cov)
cb = run(z, "coverage_tradeoff_by_borough")
piv = cb.pivot(index="design", columns="pu_borough", values="pct_valid_rows_in_adequate_cells")
display(piv)
verdict = pd.DataFrame({"overall_pct": cov.set_index("design")["pct_valid_rows_in_adequate_cells"],
                        "min_borough_pct": piv.min(axis=1)})
verdict["passes_bar"] = (verdict["overall_pct"] >= 80) & (verdict["min_borough_pct"] >= 50)
display(verdict)

,design,n_slots,n_cells,n_adequate_cells,valid_rows,pct_valid_rows_in_adequate_cells,zones_with_adequate_cell
0,A zone x weekday x hour,168,42842,5221,20205118.0,39.17,113
1,B zone x hour,24,6180,4516,20205118.0,96.49,221
2,C zone x daypart x day type,10,2579,2177,20205118.0,99.18,241
3,D zone x daypart,5,1296,1186,20205118.0,99.81,243
4,E zone (all hours),1,260,250,20205118.0,99.98,250


pu_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
design,,,,,
A zone x weekday x hour,6.82,41.10,53.10,33.76,NaN
B zone x hour,96.43,97.45,98.82,94.65,46.40
C zone x daypart x day type,99.15,99.49,99.76,98.79,84.21
D zone x daypart,99.55,99.92,99.92,99.76,98.31
E zone (all hours),100.00,99.99,100.00,99.95,99.85


,overall_pct,min_borough_pct,passes_bar
design,,,
A zone x weekday x hour,39.17,6.82,False
B zone x hour,96.49,46.40,False
C zone x daypart x day type,99.18,84.21,True
D zone x daypart,99.81,98.31,True
E zone (all hours),99.98,99.85,True


In [16]:
display(run(z, "zh_cell_size_distribution"))
display(run(z, "zh_slot_baseline"))
display(run(z, "zh_hotspot_summary").T)
display(run(z, "zh_hotspot_by_borough"))
display(run(z, "zh_staten_island_zones"))


,sample_tier,n_cells,pct_of_cells,valid_rows,pct_of_valid_rows
0,1_very_small,329,5.32,11508.0,0.06
1,2_small,1335,21.60,697538.0,3.45
2,3_adequate,4516,73.07,19496072.0,96.49


,request_hour,n_scoped,n_valid,n_excluded_negative_wait,pct_excluded,n_wait_ge10,pct_wait_ge10
0,0,665875.0,664533.0,1342.0,0.202,54505.0,8.202
1,1,465222.0,464413.0,809.0,0.174,40986.0,8.825
2,2,343416.0,342297.0,1119.0,0.326,29201.0,8.531
3,3,287746.0,283600.0,4146.0,1.441,23001.0,8.110
4,4,296903.0,285217.0,11686.0,3.936,23491.0,8.236
5,5,370494.0,356038.0,14456.0,3.902,22770.0,6.395
6,6,602474.0,588460.0,14014.0,2.326,44066.0,7.488
7,7,932357.0,911930.0,20427.0,2.191,99293.0,10.888
8,8,1072140.0,1052453.0,19687.0,1.836,55864.0,5.308
9,9,959319.0,941477.0,17842.0,1.860,23802.0,2.528


,0
n_cells_with_valid_wait,6180.00
n_adequate_cells,4516.00
n_zones_with_adequate_cell,221.00
n_hotspot_1_5x,376.00
n_hotspot_2x,175.00
n_hotspot_3x,69.00
n_hotspot_robust_2x,175.00
n_zones_with_hotspot,50.00
valid_rows_in_hotspots,1003864.00
long_waits_in_hotspots,209272.00


,pu_borough,n_zones,n_adequate_cells,zones_with_adequate_cell,n_hotspot_cells,n_robust_cells,pct_valid_rows_in_adequate_cells,pct_valid_rows_in_hotspots
0,Queens,69,1098,57,93,93,94.65,15.62
1,Staten Island,20,91,10,36,36,46.40,17.45
2,Brooklyn,61,1180,57,21,21,97.45,1.97
3,Manhattan,67,1363,61,16,16,98.82,1.93
4,Bronx,43,784,36,9,9,96.43,1.39


,PULocationID,pu_zone,zone_n_valid,pct_excluded,max_hourly_n_valid,n_adequate_hours,n_hotspot_hours,zone_pct_wait_ge10
0,206,Saint George/New Brighton,41362.0,1.138,3054,18,5,11.542
1,23,Bloomfield/Emerson Hill,31210.0,1.611,2264,18,14,13.627
2,221,Stapleton,28517.0,1.591,2207,16,4,11.432
3,156,Mariners Harbor,27750.0,1.368,2182,14,0,9.686
4,118,Heartland Village/Todt Hill,20154.0,1.688,1602,8,7,14.161
5,214,South Beach/Dongan Hills,17477.0,1.881,1182,8,0,10.185
6,115,Grymes Hill/Clifton,16851.0,1.669,1526,5,5,14.118
7,245,West Brighton,15721.0,1.738,1005,1,0,12.251
8,251,Westerleigh,15493.0,1.750,1060,1,0,9.049
9,187,Port Richmond,14876.0,1.090,1083,2,1,11.112


In [17]:
print("Top hotspot cells (zone x hour)"); display(run(z, "zh_hotspot_top_by_rate"))
print("Top hotspot cells, airport zones hidden (display aid only)"); display(run(z, "zh_hotspot_top_non_airport"))
print("Most long waits (impact)"); display(run(z, "zh_hotspot_top_by_impact"))
display(run(z, "zh_hotspot_zone_summary"))
display(run(z, "zh_hotspot_by_hour"))
display(run(z, "zh_vs_fine").T)

Top hotspot cells (zone x hour)


,pu_borough,pu_zone,request_hour,n_valid,pct_excluded,avg_trips_per_day,pct_wait_ge10,pct_wait_ge10_wilson_low,pct_ge10_if_excluded_all_short,slot_pct_wait_ge10,ratio_vs_slot,zone_pct_wait_ge10_all_hours,excess_vs_zone_pts,p50_min,p90_min,pct_uber_of_valid,hotspot_robust_flag
0,Queens,JFK Airport,23,23536,0.030,759.5,43.134,42.502,43.121,6.988,6.17,26.725,16.409,9.158,17.842,78.0,True
1,Queens,JFK Airport,0,16171,0.056,521.9,43.108,42.346,43.084,8.202,5.26,26.725,16.383,9.133,18.200,78.0,True
2,Queens,Hammels/Arverne,6,1339,3.181,44.6,43.913,41.276,42.516,7.488,5.86,23.755,20.158,9.067,18.040,73.8,True
3,Queens,JFK Airport,22,25930,0.046,836.8,38.874,38.282,38.856,6.924,5.61,26.725,12.149,8.767,15.500,78.4,True
4,Queens,Hammels/Arverne,7,2076,3.576,69.5,40.029,37.942,38.597,10.888,3.68,23.755,16.273,8.500,17.292,73.1,True
5,Queens,Far Rockaway,5,1148,3.772,38.5,39.547,36.758,38.055,6.395,6.18,20.716,18.831,8.008,17.905,72.5,True
6,Queens,JFK Airport,21,25757,0.031,831.1,35.183,34.602,35.172,6.154,5.72,26.725,8.458,8.350,14.557,79.1,True
7,Queens,JFK Airport,20,25628,0.039,827.0,34.837,34.256,34.823,4.375,7.96,26.725,8.112,8.283,14.417,79.9,True
8,Queens,Far Rockaway,6,1984,2.935,65.9,32.913,30.880,31.947,7.488,4.40,20.716,12.197,7.683,16.262,73.5,True
9,Staten Island,Grymes Hill/Clifton,7,1249,2.650,41.4,32.826,30.278,31.956,10.888,3.01,14.118,18.708,8.000,14.640,69.0,True


Top hotspot cells, airport zones hidden (display aid only)


,pu_borough,pu_zone,request_hour,n_valid,pct_excluded,avg_trips_per_day,pct_wait_ge10,pct_wait_ge10_wilson_low,pct_ge10_if_excluded_all_short,slot_pct_wait_ge10,ratio_vs_slot,zone_pct_wait_ge10_all_hours,excess_vs_zone_pts,p50_min,p90_min,pct_uber_of_valid,hotspot_robust_flag
0,Queens,Hammels/Arverne,6,1339,3.181,44.6,43.913,41.276,42.516,7.488,5.86,23.755,20.158,9.067,18.040,73.8,True
1,Queens,Hammels/Arverne,7,2076,3.576,69.5,40.029,37.942,38.597,10.888,3.68,23.755,16.273,8.500,17.292,73.1,True
2,Queens,Far Rockaway,5,1148,3.772,38.5,39.547,36.758,38.055,6.395,6.18,20.716,18.831,8.008,17.905,72.5,True
3,Queens,Far Rockaway,6,1984,2.935,65.9,32.913,30.880,31.947,7.488,4.40,20.716,12.197,7.683,16.262,73.5,True
4,Staten Island,Grymes Hill/Clifton,7,1249,2.650,41.4,32.826,30.278,31.956,10.888,3.01,14.118,18.708,8.000,14.640,69.0,True
5,Queens,Far Rockaway,7,3146,2.691,104.3,31.278,29.681,30.436,10.888,2.87,20.716,10.562,7.442,15.092,75.0,True
6,Queens,Hammels/Arverne,22,1446,0.959,47.1,31.535,29.192,31.233,6.924,4.55,23.755,7.780,7.175,15.583,82.5,True
7,Queens,Far Rockaway,14,2677,1.545,87.7,29.324,27.630,28.871,4.525,6.48,20.716,8.608,7.367,14.867,77.8,True
8,Brooklyn,Starrett City,7,1716,4.508,58.0,29.429,27.321,28.102,10.888,2.70,8.365,21.064,7.783,13.800,72.7,True
9,Staten Island,Stapleton,7,1719,2.936,57.1,29.145,27.045,28.289,10.888,2.68,11.432,17.713,7.567,14.157,70.6,True


Most long waits (impact)


,pu_borough,pu_zone,request_hour,n_valid,n_wait_ge10,pct_excluded,avg_trips_per_day,pct_wait_ge10,slot_pct_wait_ge10,ratio_vs_slot,hotspot_flag
0,Queens,JFK Airport,23,23536,10152,0.030,759.5,43.134,6.988,6.17,True
1,Queens,JFK Airport,22,25930,10080,0.046,836.8,38.874,6.924,5.61,True
2,Queens,JFK Airport,21,25757,9062,0.031,831.1,35.183,6.154,5.72,True
3,Queens,JFK Airport,20,25628,8928,0.039,827.0,34.837,4.375,7.96,True
4,Queens,LaGuardia Airport,21,25881,7601,0.177,836.4,29.369,6.154,4.77,True
5,Queens,JFK Airport,0,16171,6971,0.056,521.9,43.108,8.202,5.26,True
6,Queens,LaGuardia Airport,23,26117,6641,0.222,844.4,25.428,6.988,3.64,True
7,Queens,JFK Airport,15,24309,6543,0.070,784.7,26.916,4.153,6.48,True
8,Queens,JFK Airport,19,22950,6493,0.048,740.7,28.292,3.869,7.31,True
9,Queens,LaGuardia Airport,22,22618,6192,0.155,730.7,27.376,6.924,3.95,True


,PULocationID,pu_borough,pu_zone,n_adequate_hours,n_hotspot_hours,n_robust_hours,zone_n_valid,pct_valid_rows_in_hotspots,zone_pct_wait_ge10_all_hours
0,132,Queens,JFK Airport,24,21,21,366295.0,94.83,26.725
1,86,Queens,Far Rockaway,20,20,20,46519.0,94.66,20.716
2,117,Queens,Hammels/Arverne,18,18,18,31820.0,88.20,23.755
3,23,Staten Island,Bloomfield/Emerson Hill,18,14,14,31210.0,70.32,13.627
4,203,Queens,Rosedale,19,11,11,40659.0,51.08,12.012
5,138,Queens,LaGuardia Airport,19,7,7,360429.0,43.91,11.948
6,118,Staten Island,Heartland Village/Todt Hill,8,7,7,20154.0,49.73,14.161
7,92,Queens,Flushing,24,6,6,104418.0,28.60,7.098
8,206,Staten Island,Saint George/New Brighton,18,5,5,41362.0,20.47,11.542
9,115,Staten Island,Grymes Hill/Clifton,5,5,5,16851.0,35.31,14.118


,request_hour,n_adequate_cells,n_hotspot_cells,n_robust_cells,slot_pct_wait_ge10,pct_excluded
0,0,166,3,3,8.202,0.202
1,1,143,1,1,8.825,0.174
2,2,111,1,1,8.531,0.326
3,3,96,0,0,8.110,1.441
4,4,113,0,0,8.236,3.936
5,5,151,5,5,6.395,3.902
6,6,192,15,15,7.488,2.326
7,7,210,17,17,10.888,2.191
8,8,218,15,15,5.308,1.836
9,9,211,9,9,2.528,1.860


,0
n_coarse_hotspots,175.0
coarse_hotspots_with_a_fine_hotspot,55.0
coarse_hotspots_without_fine_hotspot,120.0
coarse_not_hotspot_but_fine_hotspot,65.0
fine_cells_total,325.0
fine_cells_inside_coarse_hotspots,239.0


In [18]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
con.execute("SELECT * FROM zh_hotspots ORDER BY PULocationID, request_hour").df().to_csv(out / "2d2_zone_hour_hotspots.csv", index=False)
cov.to_csv(out / "2d2_coverage_tradeoff.csv", index=False)
cb.to_csv(out / "2d2_coverage_tradeoff_by_borough.csv", index=False)